<a href="https://colab.research.google.com/github/Eaziwest/AI-Trader/blob/main/GilAutomation_Attendance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
pip install pandas openpyxl xlrd

In [3]:
import pandas as pd
import datetime
import warnings
from openpyxl import Workbook
from openpyxl.styles import Font

# Suppress pandas openpyxl warning
warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

def get_holidays():
    """Ask the user for holiday dates to exclude from absence calculations."""
    print("\n--- Holiday Configuration ---")
    holidays_input = input("Enter holiday dates separated by comma (YYYY-MM-DD)\n[Press Enter to skip if no holidays]: ")

    holidays = []
    if holidays_input.strip():
        try:
            holidays = [datetime.datetime.strptime(d.strip(), "%Y-%m-%d").date() for d in holidays_input.split(',')]
            print(f"Registered {len(holidays)} holiday(s).\n")
        except ValueError:
            print("Invalid date format. Please ensure you use YYYY-MM-DD. Proceeding with 0 holidays.\n")
    return holidays

def process_attendance(input_file, output_file, company_name="COMPANY"):
    holidays = get_holidays()

    print(f"\nReading '{input_file}'...")
    try:
        # 1. UNIVERSAL MULTI-SHEET FILE LOADER
        file_ext = str(input_file).lower()
        df_raw = None
        file_type = "standard"

        if file_ext.endswith(('.xls', '.xlsx')):
            engine = 'xlrd' if file_ext.endswith('.xls') else 'openpyxl'
            xls = pd.ExcelFile(input_file, engine=engine)

            # Scan all sheets in the workbook to find the right report
            for sheet in xls.sheet_names:
                temp_df = pd.read_excel(xls, sheet_name=sheet, header=None)
                # Look at a textual dump of the top left corner to identify the sheet
                text_dump = " ".join(temp_df.iloc[:6, :8].fillna("").astype(str).values.flatten())

                # EXACT MATCH required to avoid the generic "Statistical Report" sheet
                if 'Exception Statistic' in text_dump or 'First time zone' in text_dump:
                    df_raw = temp_df
                    file_type = 'exception'
                    break # Found the biometric exception report!
                elif 'ATTENDANCE SUMMARY' in text_dump:
                    df_raw = temp_df
                    file_type = 'summary'
                    break
                elif 'Employee Name' in text_dump:
                    df_raw = temp_df
                    file_type = 'standard'

            # Fallback if no signature was matched
            if df_raw is None:
                df_raw = pd.read_excel(xls, sheet_name=0, header=None)
        else:
            # Handle CSVs
            try:
                df_raw = pd.read_csv(input_file, encoding='utf-8', header=None)
            except UnicodeDecodeError:
                df_raw = pd.read_csv(input_file, encoding='latin1', header=None)

            text_dump = " ".join(df_raw.iloc[:6, :8].fillna("").astype(str).values.flatten())
            if 'Exception Statistic' in text_dump or 'First time zone' in text_dump:
                file_type = 'exception'
            elif 'ATTENDANCE SUMMARY' in text_dump:
                file_type = 'summary'

        # 2. DATA EXTRACTION BASED ON FILE TYPE
        if file_type == 'exception':
            # Find the row containing 'Name'
            header_row = df_raw[df_raw.eq('Name').any(axis=1)].index
            if len(header_row) > 0:
                start_idx = header_row[0] + 2 # Skip sub-headers like On-Duty / Off-Duty
                df = df_raw.iloc[start_idx:].copy()

                # Raw Biometric Map: Col 1 is Name, Col 2 is Dept, Col 3 is Date, Col 4 is Sign-in
                df = df[[1, 2, 3, 4]]
                df.columns = ['Employee Name', 'Department', 'Attended Date', 'Sign-In Times']
                # Clean Names: removes ' and periods (e.g. 'John.Doe -> John Doe)
                df['Employee Name'] = df['Employee Name'].astype(str).str.replace(r"^'", "", regex=True).str.replace(".", " ", regex=False)
            else:
                raise ValueError("Could not locate the 'Name' header in the Exception Report sheet.")

        elif file_type == 'summary':
            header_row = df_raw[df_raw.eq('Employee Name').any(axis=1)].index
            if len(header_row) > 0:
                start_idx = header_row[0] + 1
                df = df_raw.iloc[start_idx:].copy()
                df = df[[0, 1, 2, 3]]
                df.columns = ['Employee Name', 'Department', 'Attended Date', 'Sign-In Times']
            else:
                raise ValueError("Could not find 'Employee Name' in the Summary Report.")

        else:
            # Standard formatting
            header_row = df_raw[df_raw.eq('Employee Name').any(axis=1)].index
            if len(header_row) > 0:
                start_idx = header_row[0]
                df_raw.columns = df_raw.iloc[start_idx]
                df = df_raw.iloc[start_idx + 1:].copy()
                df = df[['Employee Name', 'Department', 'Attended Date', 'Sign-In Times']]
            else:
                raise ValueError("Could not identify standard column formats in any of the sheets.")

    except Exception as e:
        print(f"An error occurred while reading or parsing the file: {e}")
        return

    # 3. CLEANING AND PARSING DATES/TIMES
    df = df.dropna(subset=['Employee Name', 'Attended Date'])
    # Strip whitespace just in case the raw device data adds spaces to the date
    df['Attended Date'] = pd.to_datetime(df['Attended Date'].astype(str).str.strip(), errors='coerce').dt.date

    # Try parsing time formats
    df['Sign-In Times'] = pd.to_datetime(df['Sign-In Times'].astype(str).str.strip(), format='mixed', errors='coerce').dt.time
    df = df.dropna(subset=['Attended Date']) # Drop rows where Date failed to parse

    # 4. WEEK CALCULATIONS
    dates_sorted = sorted(df['Attended Date'].dropna().unique())
    if not dates_sorted:
        print("No valid dates found in data. The sheet may have been empty or misformatted.")
        return

    start_date = dates_sorted[0]
    df['Week Num'] = df['Attended Date'].apply(lambda d: ((d - start_date).days // 7) + 1)

    total_weeks = int(df['Week Num'].max())

    # 5. ANALYTICS SETUP
    late_cutoff = datetime.time(8, 15, 0)
    employee_names = sorted(df['Employee Name'].dropna().unique())

    lateness_stats = {name: {f'Week {w}': 0 for w in range(1, total_weeks + 1)} for name in employee_names}
    absence_stats = {name: {f'Week {w}': 0 for w in range(1, total_weeks + 1)} for name in employee_names}
    working_days_per_week = {}

    # Calculate Stats per week
    for w in range(1, total_weeks + 1):
        week_data = df[df['Week Num'] == w]
        if week_data.empty:
            working_days_per_week[w] = 0
            continue

        week_dates = pd.date_range(start=week_data['Attended Date'].min(),
                                   end=week_data['Attended Date'].max())

        expected_working_dates = [d.date() for d in week_dates if d.weekday() < 5 and d.date() not in holidays]
        working_days_per_week[w] = len(expected_working_dates)

        for name in employee_names:
            emp_week_data = week_data[week_data['Employee Name'] == name]

            # Count late
            late_count = 0
            for sign_in in emp_week_data['Sign-In Times'].dropna():
                if sign_in > late_cutoff:
                    late_count += 1
            lateness_stats[name][f'Week {w}'] = late_count

            # Count absent
            days_attended = emp_week_data[emp_week_data['Sign-In Times'].notna()]['Attended Date'].nunique()
            absent_count = max(0, len(expected_working_dates) - days_attended)
            absence_stats[name][f'Week {w}'] = absent_count

    # Build the Analysis Dataframe Rows
    analysis_rows = []

    for name in employee_names:
        row = [name]

        late_totals = 0
        for w in range(1, total_weeks + 1):
            val = lateness_stats[name][f'Week {w}']
            row.append(val if val > 0 else "")  # Use blank for 0 to match format
            late_totals += val
        row.append(late_totals)

        abs_totals = 0
        for w in range(1, total_weeks + 1):
            val = absence_stats[name][f'Week {w}']
            row.append(val if val > 0 else "")  # Use blank for 0 to match format
            abs_totals += val
        row.append(abs_totals)

        analysis_rows.append(row)

    # 6. OUTPUT TO EXCEL (Formatting)
    print(f"Writing formatted data to '{output_file}'...")
    wb = Workbook()
    wb.remove(wb.active) # remove default sheet

    def create_basic_sheet(sheet_title, data_df):
        ws = wb.create_sheet(sheet_title)
        ws.cell(row=1, column=1, value=f"{company_name.upper()} ATTENDANCE SUMMARY").font = Font(bold=True)
        ws.cell(row=2, column=1, value=f"Created On: {datetime.datetime.now().strftime('%m-%d-%Y')}")

        headers = ['Employee Name', 'Department', 'Attended Date', 'Sign-In Times']
        for col_num, header in enumerate(headers, 1):
            ws.cell(row=4, column=col_num, value=header).font = Font(bold=True)

        for r_idx, r_data in enumerate(data_df.itertuples(index=False), 5):
            for c_idx, value in enumerate(r_data, 1):
                cell_val = str(value) if not pd.isna(value) else ""
                ws.cell(row=r_idx, column=c_idx, value=cell_val)

        for col in ws.columns:
            max_len = max((len(str(c.value)) for c in col if c.value), default=0)
            ws.column_dimensions[col[0].column_letter].width = max_len + 4

    create_basic_sheet("RAW", df.drop(columns=['Week Num']))

    for w in range(1, total_weeks + 1):
        week_data = df[df['Week Num'] == w].drop(columns=['Week Num'])
        create_basic_sheet(f"Week {w}", week_data)

    ws_analysis = wb.create_sheet("Attendance analysis")

    ws_analysis.cell(row=1, column=1, value="Name").font = Font(bold=True)
    ws_analysis.cell(row=1, column=2, value="No. of Times Late per Week").font = Font(bold=True)
    ws_analysis.cell(row=1, column=2 + total_weeks, value="Total No. of Times late.").font = Font(bold=True)
    ws_analysis.cell(row=1, column=3 + total_weeks, value="Total Number Of Days absent/no Clock-in").font = Font(bold=True)
    ws_analysis.cell(row=1, column=4 + (total_weeks*2), value="Note: Lateness starts counting from 8.15am").font = Font(bold=True)

    for w in range(1, total_weeks + 1):
        ws_analysis.cell(row=2, column=1 + w, value=f"week {w}").font = Font(bold=True)

    for w in range(1, total_weeks + 1):
        days = working_days_per_week[w]
        ws_analysis.cell(row=2, column=2 + total_weeks + w, value=f"week {w} ({days}days)").font = Font(bold=True)

    ws_analysis.cell(row=2, column=3 + (total_weeks*2), value="Total").font = Font(bold=True)

    for r_idx, row_data in enumerate(analysis_rows, 3):
        for c_idx, value in enumerate(row_data, 1):
            ws_analysis.cell(row=r_idx, column=c_idx, value=value)

    for col in ws_analysis.columns:
        max_len = max((len(str(c.value)) for c in col if c.value), default=0)
        ws_analysis.column_dimensions[col[0].column_letter].width = max_len + 3

    wb.save(output_file)
    print(f"\nSuccessfully generated '{output_file}'!")

if __name__ == "__main__":
    print("=== Master Attendance Processor ===")

    user_input_file = input("Enter the path/name of your CSV or XLS file:\n> ").strip()

    if not user_input_file:
        print("No input file provided. Exiting script.")
    else:
        user_output_file = input("\nEnter the desired name for the Excel report (Press Enter to use 'Final_Report.xlsx'):\n> ").strip()
        if not user_output_file:
            user_output_file = "Final_Report.xlsx"
        if not user_output_file.endswith('.xlsx'):
            user_output_file += '.xlsx'

        company_name_input = input("\nEnter Company Name for headers (e.g., GIL Automation or TEAMSOURCE):\n> ").strip()
        if not company_name_input:
            company_name_input = "COMPANY"

        process_attendance(user_input_file, user_output_file, company_name_input)

=== Master Attendance Processor ===
Enter the path/name of your CSV or XLS file:
> /content/drive/MyDrive/Gil Automation/Attendance /team.xls

Enter the desired name for the Excel report (Press Enter to use 'Final_Report.xlsx'):
> TeamWay

Enter Company Name for headers (e.g., GIL Automation or TEAMSOURCE):
> Teamsource

--- Holiday Configuration ---
Enter holiday dates separated by comma (YYYY-MM-DD)
[Press Enter to skip if no holidays]: 2026-4-3
Registered 1 holiday(s).


Reading '/content/drive/MyDrive/Gil Automation/Attendance /team.xls'...
Writing formatted data to 'TeamWay.xlsx'...

Successfully generated 'TeamWay.xlsx'!
